In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


Kaggle credentials set.
Kaggle credentials successfully validated.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

dlmmdd_workshop_synthetic_source_attribution_challenge_path = kagglehub.competition_download('dlmmdd-workshop-synthetic-source-attribution-challenge')
hanifnoerrofiq_dlmmdd_91_path = kagglehub.dataset_download('hanifnoerrofiq/dlmmdd-91')

print('Data source import complete.')


100%|██████████| 10.1G/10.1G [09:16<00:00, 19.5MB/s]

Extracting files...


100%|██████████| 932M/932M [00:53<00:00, 18.2MB/s]

Extracting files...


Data source import complete.


# DLMMDD EVA02-Large + ConvNeXtV2-Large Diverse Ensemble

Calibrated ensemble of two architecturally different model families:
- **EVA02-Large 448** (ViT) 
- **ConvNeXtV2-Large 384** (CNN)

Each family uses its OWN training-matched eval transform. Per-fold temperature
calibration on processed-val OOF. Weighted log-softmax ensemble.


EVA02 (ViT) and ConvNeXtV2 (CNN) have different inductive biases. CNNs lean on
local texture filters that often survive grayscale conversion better than self-attention.
If at least ONE of your 5 wrong public-LB images is a grayscale or generator-pair
case ConvNeXtV2 reads differently, this ensemble flips it.

## Inputs you must have

1. `dlmmdd_pad_processedval_eva02_large448_96gb_outputs.zip` (EVA02-Large 3-fold ckpts)
2. `dlmmdd_robust_outputs.zip` (ConvNeXtV2-Large 3-fold ckpts) 

In [ ]:
dlmmdd_workshop_synthetic_source_attribution_challenge_path = kagglehub.competition_download(
    'dlmmdd-workshop-synthetic-source-attribution-challenge')
print('Data source import complete:', dlmmdd_workshop_synthetic_source_attribution_challenge_path)

Data source import complete: /root/.cache/kagglehub/competitions/dlmmdd-workshop-synthetic-source-attribution-challenge


In [ ]:
import os, shutil

ZIP_DEST = [
    ('/content/drive/MyDrive/dlmmdd_pad_processedval_eva02_large448_96gb_outputs.zip',
     '/content/dlmmdd_pad_processedval_eva02_large448_96gb_outputs'),
    ('/content/drive/MyDrive/dlmmdd_robust_outputs.zip',
     '/content/dlmmdd_robust_outputs'),
]
for zip_src, dst in ZIP_DEST:
    if not os.path.exists(dst):
        if os.path.exists(zip_src):
            shutil.copy(zip_src, '/content/')
            shutil.unpack_archive('/content/' + os.path.basename(zip_src), dst, 'zip')
            print('extracted', zip_src, '->', dst)
        else:
            raise FileNotFoundError(f'Missing {zip_src} on Drive. See notebook intro for prep steps.')
    else:
        print(dst, 'already present')

# Quick look at what's inside
import glob
for _, d in ZIP_DEST:
    pths = sorted(glob.glob(os.path.join(d, '**', '*.pth'), recursive=True))
    print(f'{d}: {len(pths)} ckpts')
    for p in pths[:5]: print('  ', os.path.basename(p))

extracted /content/drive/MyDrive/dlmmdd_pad_processedval_eva02_large448_96gb_outputs.zip -> /content/dlmmdd_pad_processedval_eva02_large448_96gb_outputs
extracted /content/drive/MyDrive/dlmmdd_robust_outputs.zip -> /content/dlmmdd_robust_outputs
/content/dlmmdd_pad_processedval_eva02_large448_96gb_outputs: 3 ckpts
   eva02_large_patch14_448.mim_m38m_ft_in22k_in1k_fold0.pth
   eva02_large_patch14_448.mim_m38m_ft_in22k_in1k_fold1.pth
   eva02_large_patch14_448.mim_m38m_ft_in22k_in1k_fold2.pth
/content/dlmmdd_robust_outputs: 3 ckpts
   convnextv2_large.fcmae_ft_in22k_in1k_384_fold0.pth
   convnextv2_large.fcmae_ft_in22k_in1k_384_fold1.pth
   convnextv2_large.fcmae_ft_in22k_in1k_384_fold2.pth


In [ ]:
import io, gc, glob, math, random, re, sys, atexit
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageStat, ImageEnhance, ImageFilter
from sklearn.model_selection import StratifiedKFold
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import v2

import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U',
               'timm>=1.0.20', 'huggingface_hub>=0.35.0', 'safetensors>=0.4.5'], check=False)
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True
if device.type == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
amp_dtype = torch.bfloat16 if (device.type=='cuda' and torch.cuda.is_bf16_supported()) else torch.float16
print('device', device, 'timm', timm.__version__, 'amp', amp_dtype)

device cuda timm 1.0.27 amp torch.bfloat16


In [ ]:
COMP = '/root/.cache/kagglehub/competitions/dlmmdd-workshop-synthetic-source-attribution-challenge'
TRAIN_CSV = f'{COMP}/Data/Data/training.csv'
TEST_CSV  = f'{COMP}/Data/Data/test.csv'

# Each group is a model family. transform_type: 'pad' (pad-to-square) or 'resize' (resize-stretch).
# weight: total weight given to this family in the ensemble. Will be split across
# its folds equally. The TOTAL weight (here 1.0) gets normalized in the combine step.
CKPT_GROUPS = [
    {'family': 'eva02_large_pad',
     'glob': '/content/dlmmdd_pad_processedval_eva02_large448_96gb_outputs/**/*.pth',
     'transform_type': 'pad',
     'weight': 0.75},
    {'family': 'convnextv2_large',
     'glob': '/content/dlmmdd_robust_outputs/**/*.pth',
     'transform_type': 'resize',
     'weight': 0.25},
]

FOLDS_TO_USE = [0, 1, 2]

NUM_CLASSES = 10
SEED        = 20260511
N_SPLITS    = 5
BATCH_SIZE  = 32
NUM_WORKERS = 4

# TTA per family. Both families get the same set of geometric TTAs, but with their own final-step transform (pad vs resize) so each model sees what it was trained on.
TTA_SCALES = (1.0, 0.92)
TTA_HFLIP  = True

OUT = Path('/content'); OUT.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

In [ ]:
def image_mean_color(img):
    return tuple(int(v) for v in ImageStat.Stat(img.resize((1,1), Image.Resampling.BILINEAR)).mean)
def pad_to_square_resize(img, size):
    return ImageOps.pad(img, (size, size), method=Image.Resampling.BICUBIC,
                       color=image_mean_color(img), centering=(0.5, 0.5))

class FamilyEvalTransform:
    """
    Eval transform whose final step matches the family's training distribution.
    EVA02-Large pad family -> pad_to_square_resize
    ConvNeXtV2 robust family -> Resize((size, size))   (stretches non-square; matches its training)
    """
    def __init__(self, image_size, mean, std, transform_type, scale=1.0, hflip=False):
        assert transform_type in ('pad', 'resize')
        self.image_size = image_size; self.transform_type = transform_type
        self.scale = scale; self.hflip = hflip
        self.tail = v2.Compose([
            v2.ToImage(), v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=mean, std=std),
        ])
    def __call__(self, img):
        if self.hflip: img = ImageOps.mirror(img)
        if self.scale < 1.0:
            w, h = img.size
            nw, nh = max(16, int(w*self.scale)), max(16, int(h*self.scale))
            l, t = (w-nw)//2, (h-nh)//2
            img = img.crop((l, t, l+nw, t+nh))
        if self.transform_type == 'pad':
            img = pad_to_square_resize(img, self.image_size)
        else:
            img = img.resize((self.image_size, self.image_size), Image.Resampling.BICUBIC)
        return self.tail(img)

class FamilyProcessedValTransform:
    """Same hard-processed val transform as before, with family-specific final resize/pad."""
    def __init__(self, image_size, mean, std, transform_type):
        self.image_size = image_size; self.transform_type = transform_type
        self.tail = v2.Compose([
            v2.ToImage(), v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=mean, std=std),
        ])
    def __call__(self, img):
        w,h = img.size; nw,nh = max(16,int(w*0.92)), max(16,int(h*0.92))
        l,t = (w-nw)//2, (h-nh)//2; img = img.crop((l,t,l+nw,t+nh))
        w,h = img.size
        if w>=h: img = img.crop((0,0,max(16,int(w*0.88)),h))
        else:    img = img.crop((0,0,w,max(16,int(h*0.88))))
        img = img.resize((max(32,int(img.width*0.72)),max(32,int(img.height*0.72))), Image.Resampling.BICUBIC)
        img = img.resize((max(32,int(img.width/0.72)),max(32,int(img.height/0.72))), Image.Resampling.BICUBIC)
        img = ImageEnhance.Contrast(img).enhance(0.92)
        buf = io.BytesIO(); img.save(buf,'JPEG', quality=55, optimize=False); buf.seek(0)
        img = Image.open(buf).convert('RGB')
        if self.transform_type == 'pad':
            img = pad_to_square_resize(img, self.image_size)
        else:
            img = img.resize((self.image_size, self.image_size), Image.Resampling.BICUBIC)
        return self.tail(img)

In [ ]:
def resolve_path(csv_path, is_test):
    p = str(csv_path); fn = os.path.basename(p); split = 'Test' if is_test else 'Training'
    for c in [p, f'{COMP}/{p}', f'{COMP}/Data/{p}', f'{COMP}/Data/Data/{p}',
              f'{COMP}/Data/Data/{split}/{fn}', f'{COMP}/Data/{split}/{fn}', f'{COMP}/{split}/{fn}']:
        if os.path.exists(c): return c
    return p

class SIADataset(Dataset):
    def __init__(self, df, transform, is_test):
        self.df = df.reset_index(drop=True); self.tx=transform; self.is_test=is_test
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(resolve_path(row['path'], self.is_test)).convert('RGB')
        img = self.tx(img)
        if self.is_test: return img, str(row['ID'])
        return img, int(row['y'])

# DataLoader teardown protection (silence py3.12 + PyTorch spam)
def shutdown_dataloader(loader):
    if loader is None: return
    it = getattr(loader, '_iterator', None)
    if it is not None:
        try: it._shutdown_workers()
        except Exception: pass
        loader._iterator = None

_NOISE_TOKENS = (
    'can only test a child process', '_MultiProcessingDataLoaderIter.__del__',
    'Exception ignored in:', '_shutdown_workers',
    'w.is_alive()', 'self._parent_pid == os.getpid()',
)
_orig_stderr_write = sys.stderr.write
def _filtered_stderr_write(s):
    if isinstance(s, str) and any(tok in s for tok in _NOISE_TOKENS): return len(s)
    return _orig_stderr_write(s)
sys.stderr.write = _filtered_stderr_write
atexit.register(gc.collect)

def make_loader(ds, batch_size):
    return DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS,
                     pin_memory=True, persistent_workers=False,
                     prefetch_factor=2 if NUM_WORKERS>0 else None)

@torch.no_grad()
def predict_loader(model, loader):
    model.eval(); logits_list = []; ids = []
    try:
        for batch in tqdm(loader, leave=False):
            if len(batch) == 2 and isinstance(batch[1], torch.Tensor):
                x, y = batch; ids.append(y.cpu().numpy())
            else:
                x, _ids = batch; ids.append(np.array(_ids))
            x = x.to(device, non_blocking=True, memory_format=torch.channels_last)
            with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=device.type=='cuda'):
                out = model(x)
            logits_list.append(out.float().cpu().numpy())
    finally:
        shutdown_dataloader(loader)
    return np.concatenate(logits_list, 0), np.concatenate(ids, 0)

def build_model_from_ckpt(ckpt):
    name = ckpt['model_name']
    m = timm.create_model(name, pretrained=False, num_classes=NUM_CLASSES)
    m.load_state_dict(ckpt['model'])
    m = m.to(device, memory_format=torch.channels_last).eval()
    return m, int(ckpt['image_size']), ckpt['mean'], ckpt['std']

def fit_temperature(logits, labels, max_iter=200):
    z = torch.tensor(logits, dtype=torch.float32, device=device)
    y = torch.tensor(labels, dtype=torch.long, device=device)
    logT = torch.zeros((), device=device, requires_grad=True)
    opt = torch.optim.LBFGS([logT], lr=0.1, max_iter=max_iter, line_search_fn='strong_wolfe')
    def closure():
        opt.zero_grad()
        loss = F.cross_entropy(z / logT.exp(), y)
        loss.backward(); return loss
    opt.step(closure); return float(logT.exp().item())

def parse_fold(name):
    m = re.search(r'fold(\d+)', name); return int(m.group(1)) if m else None

In [ ]:
train_df = pd.read_csv(TRAIN_CSV); test_df = pd.read_csv(TEST_CSV)
print('train', train_df.shape, 'test', test_df.shape)

kf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
fold_idx_map = {}
for fold, (tr, va) in enumerate(kf.split(train_df, train_df['y'])):
    fold_idx_map[fold] = (tr, va)
print('built fold map for seed', SEED)

train (7000, 3) test (3000, 2)
built fold map for seed 20260511


In [ ]:
# For each (family, fold) pair, compute calibrated test log-probs with TTA.
# Store along with the per-checkpoint weight so the combine step can do a weighted average.
all_logp = []        # list of (logp_array, weight, family, fold)
oof_store = {}       # (family, fold) -> (calibrated_oof_logprobs, oof_labels)  -- for the weight sweep
summary_rows = []

for group in CKPT_GROUPS:
    family = group['family']; ttype = group['transform_type']; group_w = group['weight']
    paths = sorted(glob.glob(group['glob'], recursive=True))
    paths = [p for p in paths if parse_fold(os.path.basename(p)) in FOLDS_TO_USE]
    if not paths:
        print(f'WARNING: no checkpoints found for {family}: {group["glob"]}'); continue
    per_fold_weight = group_w / len(paths)
    print(f'\n=== {family}: {len(paths)} ckpts, transform={ttype}, '
          f'group_weight={group_w}, per_fold={per_fold_weight:.4f} ===')

    for ckpt_path in paths:
        fold = parse_fold(os.path.basename(ckpt_path))
        try: ck = torch.load(ckpt_path, map_location='cpu', weights_only=False)
        except TypeError: ck = torch.load(ckpt_path, map_location='cpu')
        model, img_size, mean, std = build_model_from_ckpt(ck)
        print(f'  fold {fold} | {ck["model_name"]} {img_size}')

        # OOF -> temperature 
        _, val_idx = fold_idx_map[fold]
        oof_tx = FamilyProcessedValTransform(img_size, mean, std, ttype)
        oof_ds = SIADataset(train_df.iloc[val_idx], oof_tx, is_test=False)
        oof_z, oof_y = predict_loader(model, make_loader(oof_ds, BATCH_SIZE*2))
        oof_acc = (oof_z.argmax(1)==oof_y).mean()
        T = fit_temperature(oof_z, oof_y)
        nll0 = F.cross_entropy(torch.tensor(oof_z), torch.tensor(oof_y, dtype=torch.long)).item()
        nll1 = F.cross_entropy(torch.tensor(oof_z)/T, torch.tensor(oof_y, dtype=torch.long)).item()
        print(f'    OOF (processed-val) acc={oof_acc:.5f}  T={T:.3f}  NLL {nll0:.4f}->{nll1:.4f}')
        summary_rows.append({'family':family, 'fold':fold, 'oof_acc':oof_acc, 'T':T,
                            'weight':per_fold_weight, 'path':ckpt_path})

        # Store calibrated OOF log-probs (for the weight-sweep cell below).
        _oc = oof_z / T
        _oc = _oc - _oc.max(1, keepdims=True)
        oof_store[(family, fold)] = (_oc - np.log(np.exp(_oc).sum(1, keepdims=True)), oof_y)

        # Test TTA 
        test_logits = np.zeros((len(test_df), NUM_CLASSES), dtype=np.float64)
        n_tta = 0
        hflip_modes = (False, True) if TTA_HFLIP else (False,)
        for scale in TTA_SCALES:
            for hflip in hflip_modes:
                tx = FamilyEvalTransform(img_size, mean, std, ttype, scale=scale, hflip=hflip)
                ds = SIADataset(test_df, tx, is_test=True)
                z, _ = predict_loader(model, make_loader(ds, BATCH_SIZE*2))
                test_logits += z; n_tta += 1
                del ds; gc.collect()
        test_logits /= n_tta

        cal = test_logits / T
        cal = cal - cal.max(1, keepdims=True)
        logp = cal - np.log(np.exp(cal).sum(1, keepdims=True))
        all_logp.append((logp, per_fold_weight, family, fold))

        del model; gc.collect()
        if device.type=='cuda': torch.cuda.empty_cache()

summary_df = pd.DataFrame(summary_rows)
print('\nPer-checkpoint summary:')
print(summary_df)
print('\nMean OOF acc per family:')
print(summary_df.groupby('family')['oof_acc'].mean())



=== eva02_large_pad: 3 ckpts, transform=pad, group_weight=0.75, per_fold=0.2500 ===
  fold 0 | eva02_large_patch14_448.mim_m38m_ft_in22k_in1k 448


  0%|          | 0/22 [00:00<?, ?it/s]

    OOF (processed-val) acc=0.99500  T=0.558  NLL 0.0917->0.0229


  0%|          | 0/47 [00:00<?, ?it/s]


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    
    
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: 

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    
    
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: 

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    
    
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: 

Traceback (most recent call last):
  File "/usr/local/lib/py

...


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    
    
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: 

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    
    
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: 

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    
    
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: 

Traceback (most recent call last):
  File "/usr/local/lib/py

  0%|          | 0/47 [00:00<?, ?it/s]

  0%|          | 0/47 [00:00<?, ?it/s]

  0%|          | 0/47 [00:00<?, ?it/s]


Per-checkpoint summary:
            family  fold   oof_acc         T    weight  \
0  eva02_large_pad     0  0.995000  0.558371  0.250000   
1  eva02_large_pad     1  0.993571  0.598287  0.250000   
2  eva02_large_pad     2  0.992143  0.585643  0.250000   
3  convnextv2_large    0  0.982143  0.592716  0.083333   
4  convnextv2_large    1  0.969286  0.693213  0.083333   
5  convnextv2_large    2  0.982857  0.596168  0.083333   

                                                path  
0  /content/dlmmdd_pad_processedval_eva02_large44...  
1  /content/dlmmdd_pad_processedval_eva02_large44...  
2  /content/dlmmdd_pad_processedval_eva02_large44...  
3  /content/dlmmdd_robust_outputs/content/dlmmdd_...  
4  /content/dlmmdd_robust_outputs/content/dlmmdd_...  
5  /content/dlmmdd_robust_outputs/content/dlmmdd_...  

Mean OOF acc per family:
family
convnextv2_large    0.978095
eva02_large_pad    0.993571
Name: oof_acc, dtype: float64


In [ ]:
# OOF WEIGHT SWEEP pick the EVA02 / ConvNeXt mix that maximises held-out (out-of-fold) accuracy, instead of guessing 75/25.
# Uses the calibrated OOF log-probs stored above. Each fold's val split is the SAME images for both families (same StratifiedKFold seed), so they can be combined per fold.

families = sorted({f for _, _, f, _ in all_logp})

if len(families) < 2:
    print('Only one family present - skipping weight sweep, keeping configured weights.')
else:
    folds_present = sorted({fd for _, _, _, fd in all_logp})
    sweep = []
    for w_conv in np.round(np.arange(0.0, 0.55, 0.05), 2):
        wmap = {f: (w_conv if f == 'convnextv2_large' else 1.0 - w_conv) for f in families}
        correct = total = 0
        for fd in folds_present:
            wl = None; ys = None
            for fam in families:
                key = (fam, fd)
                if key not in oof_store:
                    continue
                lp, y = oof_store[key]
                wl = wmap[fam] * lp if wl is None else wl + wmap[fam] * lp
                ys = y
            if wl is None:
                continue
            correct += int((wl.argmax(1) == ys).sum())
            total += len(ys)
        sweep.append((float(w_conv), correct / max(1, total)))

    print('ConvNeXt weight  ->  OOF accuracy')
    for wc, acc in sweep:
        print(f'   {wc:.2f}            {acc:.5f}')

    # max() returns the FIRST best -> on ties picks the LOWEST ConvNeXt weight
    # (conservative: prefer the stronger model when the data is indifferent).
    best_w_conv, best_acc = max(sweep, key=lambda t: t[1])
    eva_only_acc = dict(sweep)[0.0]
    print(f'\n>>> OOF-optimal ConvNeXt weight = {best_w_conv:.2f}  (OOF acc {best_acc:.5f})')
    print(f'    EVA02-only OOF acc          = {eva_only_acc:.5f}')
    if best_w_conv == 0.0:
        print('    The sweep says ConvNeXt does NOT help -> ensemble collapses to EVA02-only.')
    elif best_acc - eva_only_acc < 1e-4:
        print('    ConvNeXt barely moves OOF acc -> its contribution is marginal.')

    # Rebuild all_logp weights to the OOF-optimal mix so `combine` + the
    # balancing cell automatically use the tuned weight.
    n_per_family = {f: sum(1 for _, _, ff, _ in all_logp if ff == f) for f in families}
    group_w = {f: (best_w_conv if f == 'convnextv2_large' else 1.0 - best_w_conv)
               for f in families}
    all_logp = [(lp, group_w[fam] / n_per_family[fam], fam, fd)
                for (lp, _, fam, fd) in all_logp]
    print('Rebuilt ensemble weights to the OOF-optimal mix; combine + balancing will use it.')


ConvNeXt weight  ->  OOF accuracy
   0.00            0.99357
   0.05            0.99357
   0.10            0.99357
   0.15            0.99381
   0.20            0.99381
   0.25            0.99429
   0.30            0.99452
   0.35            0.99476
   0.40            0.99548
   0.45            0.99500
   0.50            0.99476

>>> OOF-optimal ConvNeXt weight = 0.40  (OOF acc 0.99548)
    EVA02-only OOF acc          = 0.99357
Rebuilt ensemble weights to the OOF-optimal mix; combine + balancing will use it.


In [ ]:
# Weighted log-softmax average (== weighted geometric mean of softmax).
weights = np.array([w for _, w, _, _ in all_logp])
stack = np.stack([lp for lp, _, _, _ in all_logp], 0)   # (M, 3000, 10)
weighted_logp = (stack * weights[:, None, None]).sum(0) / weights.sum()
preds = weighted_logp.argmax(1)
max_p = np.exp(weighted_logp).max(1)
print(f'Mean confidence={max_p.mean():.4f}  low-confidence (<0.7) count={int((max_p<0.7).sum())}')

# Side-by-side with the EVA02-only ensemble to count disagreements.
# (Re-derive what an EVA02-only weighted average would say.)
eva_logp = [lp for lp, _, fam, _ in all_logp if fam == 'eva02_large_pad']
eva_w    = [w  for _,  w, fam, _ in all_logp if fam == 'eva02_large_pad']
if eva_logp:
    ew = np.array(eva_w)
    es = np.stack(eva_logp, 0)
    eva_only = (es * ew[:, None, None]).sum(0) / ew.sum()
    eva_preds = eva_only.argmax(1)
    diff = int((eva_preds != preds).sum())
    print(f'  vs EVA02-only ensemble: {diff} different predictions')
    diff_df = test_df[eva_preds != preds][['ID','path']].copy()
    diff_df['eva_only'] = eva_preds[eva_preds != preds]
    diff_df['combined'] = preds[eva_preds != preds]
    print(diff_df.head(20).to_string(index=False))

sub = pd.DataFrame({'ID': test_df['ID'].astype(int), 'TARGET': preds.astype(int)})
sub_path = '/content/submission_diverse_eva_convnext.csv'
sub.to_csv(sub_path, index=False)
print('saved', sub_path)
print(sub.head())

# Persist for downstream analysis
np.save('/content/diverse_logprobs.npy', weighted_logp)

# Backup to Drive
try:
    shutil.copy(sub_path, '/content/drive/MyDrive/')
    shutil.copy('/content/diverse_logprobs.npy', '/content/drive/MyDrive/')
    print('copied to Drive')
except Exception as e:
    print('drive copy skipped:', e)

Mean confidence=0.9791  low-confidence (<0.7) count=66
  vs EVA02-only ensemble: 6 different predictions
  ID               path  eva_only  combined
 685  Data/Test/685.png         8         7
3170 Data/Test/3170.png         9         4
4389 Data/Test/4389.png         9         4
8047 Data/Test/8047.png         1         2
8120 Data/Test/8120.png         4         9
9849 Data/Test/9849.png         4         9
saved /content/submission_diverse_eva_convnext.csv
   ID  TARGET
0   6       6
1  12       1
2  16       7
3  17       1
4  18       6
copied to Drive


In [ ]:
# BALANCED-MARGINAL POST-PROCESS on the diverse (EVA02 + ConvNeXtV2) ensemble

# Verified fact: the test set is exactly 300 images per class
# (10,000 total, 10 sources x 1,000, balanced; 7,000 train = 700/class).
# This enforces that marginal on the diverse-ensemble predictions.
# Produces two extra submissions: Sinkhorn (soft) and exact (hard).

from scipy.optimize import linear_sum_assignment

PER_CLASS = 300   # verified test marginal (3000 / 10)

# weighted_logp (N x 10 log-probs) comes from the `combine` cell above.
dl_logp = weighted_logp
probs = np.exp(dl_logp - dl_logp.max(1, keepdims=True))
probs = probs / probs.sum(1, keepdims=True)
N = len(probs)
raw_preds = probs.argmax(1)
ids = test_df['ID'].astype(int).to_numpy()

# current marginal of the diverse ensemble 
counts = np.bincount(raw_preds, minlength=NUM_CLASSES)
dev = counts - PER_CLASS
print('=== Diverse-ensemble class marginal (raw argmax) ===')
for c in range(NUM_CLASSES):
    flag = '' if dev[c] == 0 else f'  ({"+" if dev[c]>0 else ""}{dev[c]})'
    print(f'  class {c}: {counts[c]:4d}{flag}')
print(f'  lower-bound misclassified (sum of positive deviations) = {int(dev[dev>0].sum())}')

#  METHOD 1: Sinkhorn soft balancing
def sinkhorn_balance(p, per_class, iters=200, eps=1e-12):
    P = p.astype(np.float64).copy()
    for _ in range(iters):
        P *= per_class / (P.sum(0, keepdims=True) + eps)   # columns -> per_class
        P /= (P.sum(1, keepdims=True) + eps)               # rows -> 1
    return P

P_sink = sinkhorn_balance(probs, PER_CLASS)
sink_preds = P_sink.argmax(1)
print(f'\nSinkhorn  : changed {int((sink_preds!=raw_preds).sum())} preds; '
      f'marginal {np.bincount(sink_preds, minlength=NUM_CLASSES).tolist()}')

#  METHOD 2: exact balanced assignment (every class == 300) 
assert N == PER_CLASS * NUM_CLASSES, f'expected {PER_CLASS*NUM_CLASSES} rows, got {N}'
cost = np.empty((N, N), dtype=np.float64)
for c in range(NUM_CLASSES):
    cost[:, c*PER_CLASS:(c+1)*PER_CLASS] = -dl_logp[:, c:c+1]
print('\nsolving exact balanced assignment (3000x3000)... a few seconds')
row_ind, col_ind = linear_sum_assignment(cost)
exact_preds = np.empty(N, dtype=int)
exact_preds[row_ind] = col_ind // PER_CLASS
print(f'Exact     : changed {int((exact_preds!=raw_preds).sum())} preds; '
      f'marginal {np.bincount(exact_preds, minlength=NUM_CLASSES).tolist()}')

#  inspect what the exact method changed 
SOURCES = ['AuraFlow','Freepik','Lumina','Photon','Pixart','Playgnd2.5','SD3','SD3.5','SDXL-Turbo','Hunyuan']
changed = np.where(exact_preds != raw_preds)[0]
raw_conf = probs.max(1)
if len(changed):
    rows = []
    for i in changed:
        rows.append({'ID': int(ids[i]),
                     'raw': SOURCES[raw_preds[i]],
                     'balanced': SOURCES[exact_preds[i]],
                     'raw_conf': round(float(raw_conf[i]), 3),
                     'p_balanced': round(float(probs[i, exact_preds[i]]), 3)})
    print('\nExact-method changes (sorted by raw confidence, least confident first):')
    print(pd.DataFrame(rows).sort_values('raw_conf').to_string(index=False))
    hi = int((raw_conf[changed] > 0.90).sum())
    print(f'\n  changed preds with raw confidence > 0.90: {hi}  (want this LOW)')

# write balanced submissions
for arr, name in [(sink_preds, 'submission_diverse_balanced_sinkhorn.csv'),
                   (exact_preds, 'submission_diverse_balanced_exact.csv')]:
    s = pd.DataFrame({'ID': ids.astype(int), 'TARGET': arr.astype(int)})
    p = f'/content/{name}'
    s.to_csv(p, index=False)
    try:
        shutil.copy(p, '/content/drive/MyDrive/')
    except Exception as e:
        print('drive copy skipped:', e)
    print('saved', p)

print('\nDone. Submit submission_diverse_balanced_sinkhorn.csv first '
      '(soft balancing; refuses risky forced moves).')


=== Diverse-ensemble class marginal (raw argmax) ===
  class 0:  301  (+1)
  class 1:  300
  class 2:  300
  class 3:  300
  class 4:  303  (+3)
  class 5:  301  (+1)
  class 6:  300
  class 7:  299  (-1)
  class 8:  300
  class 9:  296  (-4)
  lower-bound misclassified (sum of positive deviations) = 5

Sinkhorn  : changed 1 preds; marginal [301, 300, 300, 300, 302, 301, 300, 299, 300, 297]

solving exact balanced assignment (3000x3000)... a few seconds
Exact     : changed 5 preds; marginal [300, 300, 300, 300, 300, 300, 300, 300, 300, 300]

Exact-method changes (sorted by raw confidence, least confident first):
  ID        raw balanced  raw_conf  p_balanced
4389     Pixart  Hunyuan     0.769       0.212
1813 Playgnd2.5    SD3.5     0.828       0.100
5734     Pixart  Hunyuan     0.875       0.100
3170     Pixart  Hunyuan     0.887       0.100
4297   AuraFlow  Hunyuan     0.888       0.035

  changed preds with raw confidence > 0.90: 0  (want this LOW)
saved /content/submission_diverse_